# 01 - Light environment of the two cultivation systems

The results generated here were used in manuscript sections 3.1 and 3.2. Five
light environments were compared: open
area (OA) at 2 m, and monoculture (MO) and agroforestry (AFS) at 2 m and 1.2 m
above the soil. Each was sampled at eleven bimonthly campaigns between
September 2003 and May 2005, over the 06:00-18:50 h diurnal period, and split
into three diurnal windows:

| Window | Clock time | Sun angle |
|---|---|---|
| Morning | 06:00-09:50 | low |
| Midday | 10:00-14:50 | near vertical |
| Afternoon | 15:00-18:50 | low |

**Produces:** Figure 1, Figure 2, Figure 3, Supplementary Figure 1,
Table 1, Supplementary Table 1.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

from yerbamate import config as C
from yerbamate import io_light, plotting as P, stats as S

P.use_paper_style()
pd.set_option("display.width", 170, "display.max_columns", 30)

ppfd = io_light.load_long(C.PPFD_LONG, "PPFD")
rfr = io_light.load_long(C.RFR_LONG, "R_FR")
print(f"PPFD readings: {len(ppfd):,}   R:FR readings: {len(rfr):,}")

## Supplementary Table 1 - light by environment and diurnal window

Mean +/- SE with the number of 10-minute readings, plus PPFD transmission and
R:FR retention expressed as a percentage of the open-area reference.

In [ ]:
def descriptives(d, window):
    g = d[d.Win == window].groupby("Env")["val"].agg(mean="mean", sem="sem", n="count")
    return g.reindex(C.ENVIRONMENTS)


def pct_of_open(d, window):
    sub = d[d.Win == window]
    open_mean = sub.loc[sub.Env == "Open_2m", "val"].mean()
    return sub.groupby("Env")["val"].mean().reindex(C.ENVIRONMENTS) / open_mean * 100


rows = []
anova = {}
for window in ["Midday", "Morning", "Afternoon"]:
    pg, rg = descriptives(ppfd, window), descriptives(rfr, window)
    tp, tr = pct_of_open(ppfd, window), pct_of_open(rfr, window)
    anova[window] = {
        "PPFD": S.one_way(ppfd[ppfd.Win == window], "val", "Env", C.ENVIRONMENTS),
        "RFR": S.one_way(rfr[rfr.Win == window], "val", "Env", C.ENVIRONMENTS),
    }
    for env in C.ENVIRONMENTS:
        rows.append({
            "Window": f"{window} ({C.WINDOW_CLOCK[window]})",
            "Environment": C.ENV_LABEL[env],
            "PPFD": f"{pg.loc[env, 'mean']:.1f} +/- {pg.loc[env, 'sem']:.1f}",
            "n_PPFD": int(pg.loc[env, "n"]),
            "Transmitted_PPFD_pct": round(tp[env], 1),
            "R:FR": f"{rg.loc[env, 'mean']:.3f} +/- {rg.loc[env, 'sem']:.3f}",
            "n_RFR": int(rg.loc[env, "n"]),
            "Transmitted_RFR_pct": round(tr[env], 1),
        })

table_s1 = pd.DataFrame(rows)
table_s1.to_csv(C.TAB_DIR / "Table_S1_light_environment.csv", index=False)
print(table_s1.to_string(index=False))

## Table 1 - two-way ANOVA (environment x measurement period)

Type-III sums of squares, the standard approach for this unbalanced design.
The partial eta-squared column generated here was carried into the manuscript;
the Residuals row is its
complement. Note that `Period` is significant for R:FR at every window but
never for PPFD - spectral quality tracks the season even where total photon
flux does not.

In [ ]:
def add_transmitted(d, window):
    """PPFD as a percentage of the same-period open-area mean."""
    sub = d[d.Win == window].copy()
    ref = sub[sub.Env == "Open_2m"].groupby("Period", observed=True)["val"].mean().rename("ref")
    sub = sub.join(ref, on="Period")
    sub["val_pct"] = sub["val"] / sub["ref"] * 100
    return sub


eta_rows = []
for window in C.WINDOWS:
    specs = [(ppfd[ppfd.Win == window], "val", "PPFD"),
             (add_transmitted(ppfd, window), "val_pct", "Transmitted PPFD (%)"),
             (rfr[rfr.Win == window], "val", "R:FR ratio")]
    for d, col, response in specs:
        t = S.two_way_eta2(d, col, "Env", "Period")
        t.insert(0, "Response", response)
        t.insert(0, "Window", window)
        eta_rows.append(t)

eta = pd.concat(eta_rows, ignore_index=True)
eta.to_csv(C.TAB_DIR / "Table_1_twoway_anova_eta2.csv", index=False)

table_1 = eta.pivot_table(index="term", columns=["Response", "Window"],
                          values="partial_eta2", sort=False)
table_1 = table_1.reindex(["Environment", "Period", "Environment x Period", "Residuals"])
table_1 = table_1.reindex(columns=pd.MultiIndex.from_product(
    [["PPFD", "Transmitted PPFD (%)", "R:FR ratio"], C.WINDOWS]))
print(table_1.round(3).to_string())

In [ ]:
# Sample sizes quoted in the Table 1 caption.
for window in C.WINDOWS:
    print(f"{window}: n = {len(ppfd[ppfd.Win == window]):,} PPFD readings")

In [ ]:
# The same models under sum-to-zero contrasts, for comparison.
invariant = []
for window in C.WINDOWS:
    for d, col, response in [(ppfd[ppfd.Win == window], "val", "PPFD"),
                             (rfr[rfr.Win == window], "val", "R:FR ratio")]:
        t = S.two_way_eta2(d, col, "Env", "Period", coding="sum")
        t.insert(0, "Response", response)
        t.insert(0, "Window", window)
        invariant.append(t)
invariant = pd.concat(invariant, ignore_index=True)
invariant.to_csv(C.TAB_DIR / "Table_1_twoway_anova_eta2_sum_contrasts.csv", index=False)

comparison = (eta.merge(invariant, on=["Window", "Response", "term"],
                        suffixes=("_analysis", "_sum_contrasts"))
                 .query("term != 'Residuals' and Response != 'Transmitted PPFD (%)'")
              [["Window", "Response", "term",
                "partial_eta2_analysis", "partial_eta2_sum_contrasts"]])
print(comparison.to_string(index=False))
print("\nThe interaction term is identical under both codings; the main effects "
      "are not.\nThe treatment-coded column generated Table 1 used in the manuscript.")

## Figure 1 - light by environment and diurnal window


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13.5, 13.5))
letters = iter("ABCDEF")
x = np.arange(len(C.ENVIRONMENTS))

for row, window in enumerate(["Midday", "Morning", "Afternoon"]):
    for col, (d, ylab, fmt) in enumerate([
            (ppfd, f"{window} PPFD (μmol m⁻² s⁻¹)", "{:.0f}"),
            (rfr, f"{window} R:FR ratio", "{:.2f}")]):
        ax = axes[row, col]
        g = descriptives(d, window)
        ax.bar(x, g["mean"], yerr=g["sem"], capsize=3,
               color=[P.ENV_COLORS[e] for e in C.ENVIRONMENTS],
               edgecolor="black", linewidth=0.7, error_kw=dict(lw=1, ecolor="#555"))

        cld = S.compact_letters(d[d.Win == window], "val", "Env", C.ENVIRONMENTS)
        for xi, (env, letter) in enumerate(zip(C.ENVIRONMENTS, cld)):
            ax.text(xi, g.loc[env, "mean"] + g.loc[env, "sem"] + g["mean"].max() * 0.03,
                    letter, ha="center", fontsize=13)

        _, p = anova[window]["PPFD" if col == 0 else "RFR"]
        ax.text(0.52, 0.93, f"P {S.fmt_p(p)}", transform=ax.transAxes,
                ha="center", fontsize=13, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels([C.ENV_LABEL[e] for e in C.ENVIRONMENTS], fontsize=12)
        ax.set_ylabel(ylab, fontsize=13.5, fontweight="bold")
        ax.set_ylim(0, g["mean"].max() * 1.30)
        if row == 2:
            ax.set_xlabel("Cultivation systems and sensor positions",
                          fontsize=13, fontweight="bold")
        P.despine(ax)
        P.panel(ax, next(letters), x=-0.10, y=1.02, size=17)

fig.tight_layout(w_pad=3.5, h_pad=2.5)
P.save(fig, "Figure_1")

## Figures 2 and 3 - two-year dynamics

Per-campaign means +/- SE for each environment, with the austral season of each
campaign marked above panel A. The three p-values are the two-way ANOVA terms
from Table 1.

The monoculture trajectory shows progressive canopy closure: the 2 m sensor
began above the developing trees and became steadily more shaded, while the
agroforestry canopy was already mature and stayed flat.

In [ ]:
SEASON_OF_PERIOD = ["Spring", "Spring", "Summer", "Autumn", "Autumn", "Winter",
                    "Spring", "Spring", "Summer", "Autumn", "Autumn"]
SEASON_COLOR = {"Spring": "#2E8B57", "Summer": "#1E7B33",
                "Autumn": "#F5A623", "Winter": "#4A90D9"}
PERIOD_SHORT = [p.replace(" 20", "-") for p in C.PERIODS]


def season_header(ax):
    """Label each run of consecutive campaigns sharing an austral season."""
    start = 0
    for i in range(1, len(SEASON_OF_PERIOD) + 1):
        if i == len(SEASON_OF_PERIOD) or SEASON_OF_PERIOD[i] != SEASON_OF_PERIOD[start]:
            season = SEASON_OF_PERIOD[start]
            ax.text((start + i - 1) / 2, 1.13, season, transform=ax.get_xaxis_transform(),
                    ha="center", va="bottom", fontsize=14, fontweight="bold",
                    color=SEASON_COLOR[season])
            start = i


def seasonal_figure(d, ylab, response, name):
    fig, axes = plt.subplots(3, 1, figsize=(13, 13.5))
    x = np.arange(len(C.PERIODS))
    for ai, window in enumerate(["Midday", "Morning", "Afternoon"]):
        ax = axes[ai]
        sub = d[d.Win == window]
        stat = sub.groupby(["Period", "Env"], observed=True)["val"].agg(["mean", "sem"])
        for env in C.ENVIRONMENTS:
            m = stat.xs(env, level="Env").reindex(C.PERIODS)
            ax.errorbar(x, m["mean"], yerr=m["sem"], fmt="-o", color=P.ENV_COLORS[env],
                        lw=2.2, ms=7, capsize=3, mec="#555", mew=0.6,
                        label=C.ENV_LABEL[env])

        terms = eta[(eta.Window == window) & (eta.Response == response)].set_index("term")
        p_env = terms.loc["Environment", "p"]
        p_per = terms.loc["Period", "p"]
        p_int = terms.loc["Environment x Period", "p"]
        ax.set_title(f"P$_{{Env}}$ {S.fmt_p(p_env)};  P$_{{Period}}$ {S.fmt_p(p_per)};  "
                     f"P$_{{Env \\times Period}}$ {S.fmt_p(p_int)}",
                     fontsize=13.5, pad=26 if ai == 0 else 8)
        ax.set_xticks(x)
        ax.set_xticklabels(PERIOD_SHORT, fontsize=12)
        ax.set_ylabel(f"{window} {ylab}", fontsize=13.5, fontweight="bold")
        ax.set_xlim(-0.4, len(C.PERIODS) - 0.6)
        P.despine(ax)
        P.panel(ax, "ABC"[ai], x=-0.055, y=1.02, size=17)
        if ai == 0:
            season_header(ax)
            ax.legend(frameon=False, fontsize=12, loc="upper right", ncol=1)
        if ai == 2:
            ax.set_xlabel("Two-year measurement periods", fontsize=13.5, fontweight="bold")
    fig.tight_layout(h_pad=2.2)
    P.save(fig, name)


seasonal_figure(ppfd, "PPFD (μmol m⁻² s⁻¹)", "PPFD", "Figure_2")
seasonal_figure(rfr, "R:FR ratio", "R:FR ratio", "Figure_3")

In [ ]:
# Per-period means behind Figures 2 and 3, saved for reference.
for window in C.WINDOWS:
    (ppfd[ppfd.Win == window].groupby(["Period", "Env"], observed=True)["val"].mean()
     .unstack("Env").reindex(C.PERIODS).round(1)
     .to_csv(C.TAB_DIR / f"monthly_PPFD_{window}.csv"))
    (rfr[rfr.Win == window].groupby(["Period", "Env"], observed=True)["val"].mean()
     .unstack("Env").reindex(C.PERIODS).round(3)
     .to_csv(C.TAB_DIR / f"monthly_RFR_{window}.csv"))
print("per-period means written to results/tables/")

## Supplementary Figure 1 - mean diurnal course

Hourly means pooled over all eleven campaigns. The dashed vertical line marks
local solar midday (~12:30 BRT); the dotted horizontal line in panel B is the
R:FR of sunlight that has not passed through vegetation (~1.1).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.4))
for ax, d, ylab, letter in [(axes[0], ppfd, "PPFD (μmol m⁻² s⁻¹)", "A"),
                            (axes[1], rfr, "R:FR ratio", "B")]:
    hourly = d[d.h.between(6, 18)].groupby(["Env", "h"])["val"].mean().reset_index()
    for env in C.ENVIRONMENTS:
        s = hourly[hourly.Env == env]
        ax.plot(s.h, s.val, "-o", color=P.ENV_COLORS[env], lw=2.2, ms=5,
                mec="#555", mew=0.5, label=C.ENV_LABEL[env])
    ax.axvline(12.5, color="#888", ls="--", lw=1.2)
    ax.set_xlabel("Hour of day (BRT)", fontsize=13.5)
    ax.set_ylabel(ylab, fontsize=13.5)
    ax.set_xlim(5.5, 18.5)
    P.despine(ax)
    P.panel(ax, letter, x=0.0, y=1.02, size=16)
axes[1].axhline(1.1, color="#C0392B", ls=":", lw=1.4)

fig.legend(handles=[Patch(color=P.ENV_COLORS[e], label=C.ENV_LABEL[e]) for e in C.ENVIRONMENTS],
           loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.03))
fig.tight_layout(rect=[0, 0.06, 1, 1])
P.save(fig, "Figure_S1")

## Key numbers quoted

In [ ]:
mid_p = descriptives(ppfd, "Midday")["mean"]
mid_r = descriptives(rfr, "Midday")["mean"]
print("Midday PPFD reduction relative to open area:")
for env in C.ENVIRONMENTS[1:]:
    print(f"  {C.ENV_LABEL[env]:10s} {mid_p[env]:7.1f}  ({100 - mid_p[env] / mid_p['Open_2m'] * 100:.1f} % reduction)")
print("\nMidday R:FR reduction relative to open area:")
for env in C.ENVIRONMENTS[1:]:
    print(f"  {C.ENV_LABEL[env]:10s} {mid_r[env]:7.3f}  ({100 - mid_r[env] / mid_r['Open_2m'] * 100:.1f} % reduction)")

In [ ]:
# Morning-afternoon asymmetry: the difference widens with canopy depth.
asym = []
for env in C.ENVIRONMENTS:
    pm = ppfd[(ppfd.Win == "Morning") & (ppfd.Env == env)]["val"].mean()
    pa = ppfd[(ppfd.Win == "Afternoon") & (ppfd.Env == env)]["val"].mean()
    rm = rfr[(rfr.Win == "Morning") & (rfr.Env == env)]["val"].mean()
    ra = rfr[(rfr.Win == "Afternoon") & (rfr.Env == env)]["val"].mean()
    asym.append({"Environment": C.ENV_LABEL[env], "PPFD_morning": round(pm, 1),
                 "PPFD_afternoon": round(pa, 1), "PPFD_ratio": round(pa / pm, 2),
                 "RFR_morning": round(rm, 3), "RFR_afternoon": round(ra, 3),
                 "RFR_difference": round(ra - rm, 3)})
asym = pd.DataFrame(asym)
asym.to_csv(C.TAB_DIR / "morning_afternoon_asymmetry.csv", index=False)
print(asym.to_string(index=False))

In [ ]:
# Supplementary Table 1 values.
assert abs(mid_p["Open_2m"] - 1245.5) < 0.1
assert abs(mid_p["AFS_1.2m"] - 62.3) < 0.1
assert abs(mid_r["AFS_1.2m"] - 0.606) < 0.001

# Table 1 values generated by the analysis and used in the manuscript.
ANALYSIS_TABLE_1 = {
    ("PPFD", "Morning"): (0.135, 0.000, 0.146), ("PPFD", "Midday"): (0.462, 0.003, 0.502),
    ("PPFD", "Afternoon"): (0.121, 0.001, 0.159),
    ("Transmitted PPFD (%)", "Morning"): (0.076, 0.001, 0.042),
    ("Transmitted PPFD (%)", "Midday"): (0.271, 0.002, 0.244),
    ("Transmitted PPFD (%)", "Afternoon"): (0.048, 0.001, 0.059),
    ("R:FR ratio", "Morning"): (0.234, 0.063, 0.298),
    ("R:FR ratio", "Midday"): (0.179, 0.048, 0.271),
    ("R:FR ratio", "Afternoon"): (0.244, 0.072, 0.322),
}
for (response, window), expected in ANALYSIS_TABLE_1.items():
    got = (eta[(eta.Response == response) & (eta.Window == window)]
           .set_index("term")["partial_eta2"])
    for term, want in zip(["Environment", "Period", "Environment x Period"], expected):
        assert abs(got[term] - want) < 0.0011, \
            f"{response} {window} {term}: {got[term]} vs validated analysis value {want}"
print("Validated the Figure 1-3, S1 and Table 1, S1 results used in the manuscript.")